# Ex7 — MNIST MLP & CNN
Run on Google Colab: **Runtime → Change runtime type → T4 GPU**

In [ ]:
!pip install -q tabulate torch torchvision tensorboard scikit-learn matplotlib

## 1. Imports & Device

In [ ]:
import os, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tabulate import tabulate
from torch import Generator
from torch.utils.data import DataLoader, Subset, random_split
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Dataset

In [ ]:
def get_mnist_dataset():
    transform = transforms.ToTensor()
    train_dataset = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
    test_dataset  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
    return train_dataset, test_dataset

def get_loaders(batch_size=64, val_ratio=0.2, seed=42):
    whole_dataset, test_dataset = get_mnist_dataset()
    train_ds, val_ds = random_split(whole_dataset, [1 - val_ratio, val_ratio],
                                    generator=Generator().manual_seed(seed))
    return (DataLoader(train_ds,    batch_size=batch_size, shuffle=True),
            DataLoader(val_ds,      batch_size=batch_size, shuffle=False),
            DataLoader(test_dataset, batch_size=batch_size, shuffle=False))

def get_cross_validation_loaders(batch_size=64, folds=5, seed=42):
    whole_dataset, test_dataset = get_mnist_dataset()
    idxs = list(range(len(whole_dataset)))
    np.random.seed(seed)
    np.random.shuffle(idxs)
    fold_size    = len(whole_dataset) // folds
    fold_indices = [idxs[i:i + fold_size] for i in range(0, len(idxs), fold_size)]
    for i in range(folds):
        val_idx   = fold_indices[i]
        train_idx = np.concatenate([fold_indices[j] for j in range(folds) if j != i])
        yield (DataLoader(Subset(whole_dataset, train_idx), batch_size=batch_size, shuffle=True),
               DataLoader(Subset(whole_dataset, val_idx),  batch_size=batch_size, shuffle=False),
               DataLoader(test_dataset,                     batch_size=batch_size, shuffle=False))

print("Dataset functions ready.")

## 3. Models

In [ ]:
class Architecture:
    def __init__(self, input_dim, hidden_dims, output_dim, dropout=None):
        self.input_dim   = input_dim
        self.hidden_dims = hidden_dims
        self.output_dim  = output_dim
        self.dropout     = dropout if dropout is not None else [0.0] * len(hidden_dims)

    def validate(self):
        assert self.input_dim > 0 and self.output_dim > 0
        assert len(self.hidden_dims) == len(self.dropout)
        for d in self.hidden_dims: assert d > 0
        for d in self.dropout:     assert 0.0 <= d < 1.0


class SimpleMLP(nn.Module):
    def __init__(self, arch):
        super().__init__()
        arch.validate()
        layers = [nn.Flatten()]
        for i in range(len(arch.hidden_dims)):
            in_dim = arch.hidden_dims[i-1] if i > 0 else arch.input_dim
            layers += [nn.Linear(in_dim, arch.hidden_dims[i]), nn.ReLU()]
            if i < len(arch.hidden_dims) - 1:
                layers.append(nn.Dropout(arch.dropout[i]))
        layers.append(nn.Linear(arch.hidden_dims[-1], arch.output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x): return self.net(x)


class SimpleCNN(nn.Module):
    def __init__(self, arch):
        super().__init__()
        arch.validate()
        layers, pool_count = [], 0
        for i in range(len(arch.hidden_dims)):
            in_ch = arch.hidden_dims[i-1] if i > 0 else arch.input_dim
            layers += [nn.Conv2d(in_ch, arch.hidden_dims[i], 3, padding=1), nn.ReLU()]
            if pool_count < 3:
                layers.append(nn.MaxPool2d(2))
                pool_count += 1
            if i < len(arch.hidden_dims) - 1:
                layers.append(nn.Dropout2d(arch.dropout[i]))
        spatial = (28 // (2 ** pool_count)) ** 2
        layers += [nn.Flatten(), nn.Linear(arch.hidden_dims[-1] * spatial, arch.output_dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, x): return self.net(x)


print("Model classes ready.")

## 4. Training utilities

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = total_correct = total_count = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss   = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss    += loss.item() * len(X)
        total_correct += (logits.argmax(1) == y).sum().item()
        total_count   += len(X)
    return total_loss / total_count, total_correct / total_count


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = total_correct = total_count = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            loss   = criterion(logits, y)
            total_loss    += loss.item() * len(X)
            total_correct += (logits.argmax(1) == y).sum().item()
            total_count   += len(X)
    return total_loss / total_count, total_correct / total_count


def train_loop(model, train_loader, val_loader, criterion, optimizer, device,
               num_epochs=20, log_dir="logs"):
    writer = SummaryWriter(log_dir)
    last_val_acc = no_improve = 0
    for epoch in range(num_epochs):
        t_loss, t_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        v_loss, v_acc = evaluate(model, val_loader, criterion, device)
        print(f"Epoch {epoch+1}/{num_epochs} | Train {t_loss:.4f}/{t_acc:.4f} | Val {v_loss:.4f}/{v_acc:.4f}")
        writer.add_scalars("Loss",     {"Train": t_loss, "Val": v_loss}, epoch)
        writer.add_scalars("Accuracy", {"Train": t_acc,  "Val": v_acc},  epoch)
        if last_val_acc >= v_acc:
            no_improve += 1
        else:
            no_improve, last_val_acc = 0, v_acc
        if no_improve >= max(1, num_epochs // 4):
            print("Early stopping triggered")
            yield t_loss, t_acc, v_loss, v_acc
            break
        yield t_loss, t_acc, v_loss, v_acc
    writer.close()


def get_metrics(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            all_preds.append(model(X).argmax(1).cpu())
            all_labels.append(y.cpu())
    all_preds  = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    acc = (all_preds == all_labels).float().mean().item()
    cm  = confusion_matrix(all_labels.numpy(), all_preds.numpy())
    sensitivity, specificity = [], []
    for i in range(len(cm)):
        TP = cm[i, i];  FN = cm[i].sum() - TP
        FP = cm[:, i].sum() - TP;  TN = cm.sum() - (TP + FN + FP)
        sensitivity.append(TP / (TP + FN) if (TP + FN) > 0 else 0.0)
        specificity.append(TN / (TN + FP) if (TN + FP) > 0 else 0.0)
    return acc, cm, sensitivity, specificity


print("Training utilities ready.")

## 5. Plotting utilities

In [ ]:
def plot_metrics(t_loss, t_acc, v_loss, v_acc, title=""):
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes[0][0].plot(t_loss);  axes[0][0].set_title("Train Loss");        axes[0][0].grid(True)
    axes[0][1].plot(t_acc,  color="orange"); axes[0][1].set_title("Train Accuracy");     axes[0][1].grid(True)
    axes[1][0].plot(v_loss);  axes[1][0].set_title("Validation Loss");   axes[1][0].grid(True)
    axes[1][1].plot(v_acc,  color="orange"); axes[1][1].set_title("Validation Accuracy"); axes[1][1].grid(True)
    if title: fig.suptitle(title, fontsize=13)
    fig.tight_layout(); plt.show()

def plot_confusion(cm, title="Confusion Matrix"):
    ConfusionMatrixDisplay(confusion_matrix=cm).plot(cmap=plt.cm.Blues)
    plt.title(title); plt.show()

def predict_samples(model, device, seed=42):
    rng = np.random.default_rng(seed)
    test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=transforms.ToTensor())
    targets = np.array(test_ds.targets)
    selected = [int(rng.choice(np.where(targets == c)[0])) for c in range(10)]
    model.eval()
    fig, axes = plt.subplots(2, 10, figsize=(20, 5))
    with torch.no_grad():
        for col, idx in enumerate(selected):
            img, true_lbl = test_ds[idx]
            probs     = F.softmax(model(img.unsqueeze(0).to(device)), dim=1).squeeze().cpu().numpy()
            pred_lbl  = int(probs.argmax())
            axes[0][col].imshow(img.squeeze(), cmap="gray")
            axes[0][col].set_title(f"T:{true_lbl}
P:{pred_lbl}",
                                   color="green" if pred_lbl == true_lbl else "red", fontsize=9)
            axes[0][col].axis("off")
            axes[1][col].bar(range(10), probs, color="steelblue")
            axes[1][col].set_xticks(range(10)); axes[1][col].set_ylim(0, 1)
            axes[1][col].tick_params(labelsize=7)
    axes[1][0].set_ylabel("Prob")
    fig.suptitle("Sample Predictions (green=correct, red=wrong)")
    fig.tight_layout(); plt.show()

print("Plotting utilities ready.")

## 6. Experiment runner

In [ ]:
def run_experiment(model_type, hidden_dims, dropout, num_epochs=20,
                   lr=0.001, batch_size=64, n_runs=5, label="exp"):
    val_accs, val_losses      = [], []
    best_val_acc, best_model  = 0.0, None
    best_curves, best_speed   = None, float("inf")
    test_loader               = None

    for i in range(n_runs):
        print(f"
--- {label} | Run {i+1}/{n_runs} ---")
        if model_type == "mlp":
            model = SimpleMLP(Architecture(784, hidden_dims, 10, dropout)).to(device)
        else:
            model = SimpleCNN(Architecture(1,   hidden_dims, 10, dropout)).to(device)
        criterion    = nn.CrossEntropyLoss()
        optimizer    = torch.optim.Adam(model.parameters(), lr=lr)
        train_loader, val_loader, test_loader = get_loaders(batch_size=batch_size)

        t_losses, v_losses, t_accs, v_accs = [], [], [], []
        start = time.time()
        for t_loss, t_acc, v_loss, v_acc in train_loop(
                model, train_loader, val_loader, criterion, optimizer, device,
                num_epochs=num_epochs, log_dir=f"logs/{label}_run{i+1}"):
            t_losses.append(t_loss); v_losses.append(v_loss)
            t_accs.append(t_acc);   v_accs.append(v_acc)
        best_speed = min(best_speed, time.time() - start)

        val_accs.append(v_accs[-1]); val_losses.append(v_losses[-1])
        if v_accs[-1] > best_val_acc:
            best_val_acc = v_accs[-1]
            best_model   = model
            best_curves  = (t_losses, t_accs, v_losses, v_accs)

    test_acc, cm, sens, spec = get_metrics(best_model, test_loader, device)
    results = {
        "val_acc_min":  min(val_accs),  "val_acc_max":  max(val_accs),
        "val_acc_avg":  sum(val_accs)/len(val_accs),
        "val_loss_min": min(val_losses), "val_loss_max": max(val_losses),
        "val_loss_avg": sum(val_losses)/len(val_losses),
        "test_acc": test_acc, "best_time_s": best_speed,
        "cm": cm, "sensitivity": sens, "specificity": spec,
    }
    print(f"
=== {label} ===")
    print(tabulate([[results["val_acc_min"], results["val_acc_avg"], results["val_acc_max"],
                     results["test_acc"], results["best_time_s"]]],
                   headers=["Val Min","Val Avg","Val Max","Test Acc","Best Time(s)"],
                   tablefmt="grid", floatfmt=".4f"))
    return best_model, results, best_curves

print("run_experiment() ready.")

## 7. MLP Experiments (≥2 architectures × 5 runs)

In [ ]:
mlp1_model, mlp1_res, mlp1_curves = run_experiment(
    "mlp", [256, 128], [0.0, 0.0], num_epochs=20, label="mlp_arch1")
plot_metrics(*mlp1_curves, title="MLP Arch1 — Best Run")
plot_confusion(mlp1_res["cm"], "MLP Arch1")

In [ ]:
mlp2_model, mlp2_res, mlp2_curves = run_experiment(
    "mlp", [512, 256, 128], [0.0, 0.0, 0.0], num_epochs=20, label="mlp_arch2")
plot_metrics(*mlp2_curves, title="MLP Arch2 — Best Run")
plot_confusion(mlp2_res["cm"], "MLP Arch2")

## 8. CNN Experiments (≥3 architectures × 5 runs)

In [ ]:
cnn1_model, cnn1_res, cnn1_curves = run_experiment(
    "cnn", [32, 64], [0.0, 0.0], num_epochs=20, label="cnn_arch1")
plot_metrics(*cnn1_curves, title="CNN Arch1 — Best Run")
plot_confusion(cnn1_res["cm"], "CNN Arch1")

In [ ]:
cnn2_model, cnn2_res, cnn2_curves = run_experiment(
    "cnn", [64, 128], [0.0, 0.0], num_epochs=20, label="cnn_arch2")
plot_metrics(*cnn2_curves, title="CNN Arch2 — Best Run")
plot_confusion(cnn2_res["cm"], "CNN Arch2")

In [ ]:
cnn3_model, cnn3_res, cnn3_curves = run_experiment(
    "cnn", [32, 64, 128], [0.0, 0.0, 0.0], num_epochs=20, label="cnn_arch3")
plot_metrics(*cnn3_curves, title="CNN Arch3 — Best Run")
plot_confusion(cnn3_res["cm"], "CNN Arch3")

## 9. Dropout Experiments (≥3 settings × 5 runs)

In [ ]:
do0_model, do0_res, do0_curves = run_experiment(
    "cnn", [64, 128], [0.0, 0.0], num_epochs=30, label="dropout_00")
plot_metrics(*do0_curves, title="Dropout 0.0")

In [ ]:
do2_model, do2_res, do2_curves = run_experiment(
    "cnn", [64, 128], [0.2, 0.2], num_epochs=30, label="dropout_02")
plot_metrics(*do2_curves, title="Dropout 0.2")

In [ ]:
do5_model, do5_res, do5_curves = run_experiment(
    "cnn", [64, 128], [0.5, 0.5], num_epochs=30, label="dropout_05")
plot_metrics(*do5_curves, title="Dropout 0.5")

## 10. CPU vs GPU Training Time

In [ ]:
def measure_time(model_type, hidden_dims, dropout, target_device, num_epochs=5):
    if model_type == "mlp":
        model = SimpleMLP(Architecture(784, hidden_dims, 10, dropout)).to(target_device)
    else:
        model = SimpleCNN(Architecture(1,   hidden_dims, 10, dropout)).to(target_device)
    crit = nn.CrossEntropyLoss()
    opt  = torch.optim.Adam(model.parameters(), lr=0.001)
    train_loader, val_loader, _ = get_loaders()
    start = time.time()
    for _ in train_loop(model, train_loader, val_loader, crit, opt,
                        target_device, num_epochs=num_epochs, log_dir="logs/timing"):
        pass
    return time.time() - start

cpu_time = measure_time("cnn", [64, 128], [0.0, 0.0], torch.device("cpu"))
print(f"CPU time: {cpu_time:.2f}s")

if torch.cuda.is_available():
    gpu_time = measure_time("cnn", [64, 128], [0.0, 0.0], torch.device("cuda"))
    print(f"GPU time: {gpu_time:.2f}s")
    print(f"Speedup : {cpu_time / gpu_time:.1f}x")
else:
    print("GPU not available — enable in Colab: Runtime → Change runtime type → T4 GPU")

## 11. MLP vs CNN Comparison Summary

In [ ]:
rows = []
for label, res in [("MLP Arch1", mlp1_res), ("MLP Arch2", mlp2_res),
                   ("CNN Arch1", cnn1_res), ("CNN Arch2", cnn2_res), ("CNN Arch3", cnn3_res)]:
    rows.append([label, f"{res['val_acc_min']:.4f}", f"{res['val_acc_avg']:.4f}",
                 f"{res['val_acc_max']:.4f}", f"{res['test_acc']:.4f}", f"{res['best_time_s']:.1f}s"])
print(tabulate(rows,
    headers=["Model","Val Min","Val Avg","Val Max","Test Acc","Best Time"],
    tablefmt="grid"))

## 12. Sample Predictions — Best MLP & CNN

In [ ]:
predict_samples(mlp1_model, device)
predict_samples(cnn2_model, device)